In [1]:
!apt-get update -qq
!apt-get install -y ffmpeg
!pip install -q flask pyngrok transformers torch openai-whisper requests


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 76 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.4 MB/s eta 0:00:00


In [14]:
NGROK_AUTH_TOKEN = "39nw0VpNZrg7xIhPJSqERtcTBro_7jBbDDeT8UuANtgp9TxBy"
N8N_WEBHOOK_URL = "https://asmaamamdouh2005.app.n8n.cloud/webhook-test/crm-result"
print("N8N webhook updated:", N8N_WEBHOOK_URL)
TEXT_MODEL_NAME = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
WHISPER_MODEL_NAME = "base"

UPLOAD_FOLDER = "/content/uploads"
DB_PATH = "/content/crm_sentiment.db"
MAX_CONTENT_LENGTH_MB = 25


N8N webhook updated: https://asmaamamdouh2005.app.n8n.cloud/webhook-test/crm-result


In [3]:
import os
import uuid
import sqlite3
from datetime import datetime, timezone
from threading import Thread

import requests
import whisper
from flask import Flask, jsonify, request, render_template_string
from pyngrok import ngrok
from transformers import pipeline
from werkzeug.utils import secure_filename


In [4]:
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

ALLOWED_AUDIO_EXTENSIONS = {"wav", "mp3", "m4a", "ogg", "webm", "mp4", "mpeg"}

print("Loading sentiment model...")
classifier = pipeline("sentiment-analysis", model=TEXT_MODEL_NAME)

print("Loading whisper model...")
speech_model = whisper.load_model(WHISPER_MODEL_NAME)

print("Models loaded successfully.")


Loading sentiment model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Loading whisper model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 110MiB/s]


Models loaded successfully.


In [5]:
def get_db_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS customer_interactions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            customer_name TEXT NOT NULL,
            input_type TEXT NOT NULL,
            original_text TEXT,
            transcribed_text TEXT,
            sentiment TEXT NOT NULL,
            sentiment_ar TEXT NOT NULL,
            confidence REAL NOT NULL,
            recommended_action TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def allowed_audio_file(filename: str):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_AUDIO_EXTENSIONS


def map_sentiment(label: str):
    value = (label or "").lower().strip()

    if "positive" in value:
        return "Happy", "سعيد"
    elif "negative" in value:
        return "Angry", "غاضب"
    else:
        return "Neutral", "محايد"


def recommended_action(sentiment: str):
    mapping = {
        "Angry": "Escalate immediately to customer support",
        "Happy": "Thank the customer and continue engagement",
        "Neutral": "Monitor conversation and collect more context",
    }
    return mapping.get(sentiment, "Manual review")


def analyze_text_message(text: str):
    result = classifier(text)[0]
    sentiment, sentiment_ar = map_sentiment(result.get("label"))

    return {
        "sentiment": sentiment,
        "sentiment_ar": sentiment_ar,
        "confidence": round(float(result.get("score", 0.0)), 4),
        "recommended_action": recommended_action(sentiment),
    }


def transcribe_audio(audio_path: str):
    result = speech_model.transcribe(audio_path, fp16=False)
    return (result.get("text") or "").strip()


def save_result(payload: dict):
    conn = get_db_connection()
    cursor = conn.execute("""
        INSERT INTO customer_interactions
        (customer_name, input_type, original_text, transcribed_text, sentiment, sentiment_ar, confidence, recommended_action, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        payload["customer_name"],
        payload["input_type"],
        payload.get("original_text"),
        payload.get("transcribed_text"),
        payload["sentiment"],
        payload["sentiment_ar"],
        payload["confidence"],
        payload["recommended_action"],
        payload["created_at"],
    ))
    conn.commit()
    row_id = cursor.lastrowid
    conn.close()
    return row_id


def notify_n8n(payload: dict):
    global N8N_WEBHOOK_URL

    if not N8N_WEBHOOK_URL or "PUT_" in N8N_WEBHOOK_URL:
        return {"sent": False, "message": "N8N webhook not configured yet"}

    try:
        response = requests.post(N8N_WEBHOOK_URL, json=payload, timeout=10)
        return {
            "sent": True,
            "status_code": response.status_code
        }
    except Exception as e:
        return {
            "sent": False,
            "message": str(e)
        }


def build_text_payload(customer_name: str, text: str):
    analysis = analyze_text_message(text)

    return {
        "customer_name": customer_name,
        "input_type": "text",
        "original_text": text,
        "transcribed_text": text,
        **analysis,
        "created_at": now_iso(),
    }


def build_voice_payload(customer_name: str, file_storage):
    if not file_storage or file_storage.filename == "":
        raise ValueError("Audio file is required")

    if not allowed_audio_file(file_storage.filename):
        raise ValueError("Unsupported audio format")

    ext = file_storage.filename.rsplit(".", 1)[1].lower()
    safe_name = secure_filename(file_storage.filename)
    temp_name = f"{uuid.uuid4().hex}_{safe_name.rsplit('.', 1)[0]}.{ext}"
    temp_path = os.path.join(UPLOAD_FOLDER, temp_name)

    file_storage.save(temp_path)

    try:
        text = transcribe_audio(temp_path)
        if not text:
            raise ValueError("Could not transcribe audio")

        analysis = analyze_text_message(text)

        return {
            "customer_name": customer_name,
            "input_type": "voice",
            "original_text": None,
            "transcribed_text": text,
            **analysis,
            "created_at": now_iso(),
        }
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


init_db()
print("Database initialized.")


Database initialized.


In [6]:
HOME_HTML = """
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>CRM Sentiment Analysis</title>
  <style>
    body {
      font-family: Arial, sans-serif;
      background: #f5f7fb;
      margin: 0;
      padding: 0;
      color: #222;
    }
    .container {
      width: 92%;
      max-width: 1100px;
      margin: 30px auto;
    }
    h1 {
      text-align: center;
      color: #0b4f8a;
      margin-bottom: 8px;
    }
    .subtitle {
      text-align: center;
      color: #555;
      margin-bottom: 30px;
    }
    .grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 20px;
    }
    .card {
      background: white;
      border-radius: 16px;
      padding: 20px;
      box-shadow: 0 4px 15px rgba(0,0,0,0.08);
    }
    label {
      display: block;
      margin-top: 12px;
      margin-bottom: 6px;
      font-weight: bold;
    }
    input, textarea, button {
      width: 100%;
      box-sizing: border-box;
      padding: 12px;
      border-radius: 10px;
      border: 1px solid #ccc;
      font-size: 15px;
    }
    textarea {
      min-height: 120px;
      resize: vertical;
    }
    button {
      margin-top: 16px;
      background: #0b4f8a;
      color: white;
      border: none;
      cursor: pointer;
      font-weight: bold;
    }
    button:hover {
      background: #083b67;
    }
    .result {
      margin-top: 25px;
      background: #111827;
      color: #e5e7eb;
      border-radius: 14px;
      padding: 18px;
      white-space: pre-wrap;
      overflow-x: auto;
      min-height: 120px;
    }
    @media (max-width: 900px) {
      .grid {
        grid-template-columns: 1fr;
      }
    }
  </style>
</head>
<body>
  <div class="container">
    <h1>CRM Sentiment Analysis</h1>
    <p class="subtitle">تحليل مشاعر العميل من النص أو الصوت</p>

    <div class="grid">
      <div class="card">
        <h2>تحليل رسالة نصية</h2>
        <label>اسم العميل</label>
        <input type="text" id="textCustomerName" placeholder="مثال: Ahmed Ali" />

        <label>رسالة العميل</label>
        <textarea id="textMessage" placeholder="اكتب رسالة العميل هنا..."></textarea>

        <button onclick="analyzeText()">تحليل النص</button>
      </div>

      <div class="card">
        <h2>تحليل رسالة صوتية</h2>
        <label>اسم العميل</label>
        <input type="text" id="voiceCustomerName" placeholder="مثال: Sara Mohamed" />

        <label>ملف الصوت</label>
        <input type="file" id="audioFile" accept=".wav,.mp3,.m4a,.ogg,.webm,.mp4,.mpeg" />

        <button onclick="analyzeVoice()">تحليل الصوت</button>
      </div>
    </div>

    <div class="result" id="resultBox">النتيجة ستظهر هنا...</div>
  </div>

  <script>
    const resultBox = document.getElementById("resultBox");

    function showResult(data) {
      resultBox.textContent = JSON.stringify(data, null, 2);
    }

    async function analyzeText() {
      const customer_name = document.getElementById("textCustomerName").value.trim();
      const text = document.getElementById("textMessage").value.trim();

      if (!text) {
        alert("من فضلك اكتبي رسالة العميل");
        return;
      }

      resultBox.textContent = "جاري تحليل النص...";

      try {
        const response = await fetch("/analyze_text", {
          method: "POST",
          headers: {
            "Content-Type": "application/json"
          },
          body: JSON.stringify({ customer_name, text })
        });

        const data = await response.json();
        showResult(data);
      } catch (error) {
        showResult({ ok: false, error: error.message });
      }
    }

    async function analyzeVoice() {
      const customer_name = document.getElementById("voiceCustomerName").value.trim();
      const audioFile = document.getElementById("audioFile").files[0];

      if (!audioFile) {
        alert("من فضلك ارفعي ملف صوت");
        return;
      }

      resultBox.textContent = "جاري تحليل الصوت...";

      try {
        const formData = new FormData();
        formData.append("customer_name", customer_name);
        formData.append("audio", audioFile);

        const response = await fetch("/analyze_voice", {
          method: "POST",
          body: formData
        });

        const data = await response.json();
        showResult(data);
      } catch (error) {
        showResult({ ok: false, error: error.message });
      }
    }
  </script>
</body>
</html>
"""

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = MAX_CONTENT_LENGTH_MB * 1024 * 1024


@app.route("/")
def home():
    return render_template_string(HOME_HTML)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "ok": True,
        "message": "CRM Sentiment API is running"
    })


@app.route("/history", methods=["GET"])
def history():
    limit = min(int(request.args.get("limit", 10)), 50)
    conn = get_db_connection()
    rows = conn.execute(
        "SELECT * FROM customer_interactions ORDER BY id DESC LIMIT ?",
        (limit,)
    ).fetchall()
    conn.close()

    return jsonify([dict(row) for row in rows])


@app.route("/analyze_text", methods=["POST"])
def analyze_text_api():
    try:
        data = request.get_json(silent=True) or request.form.to_dict()
        customer_name = (data.get("customer_name") or "Unknown Customer").strip()
        text = (data.get("text") or "").strip()

        if not text:
            return jsonify({"ok": False, "error": "Text is required"}), 400

        payload = build_text_payload(customer_name, text)
        record_id = save_result(payload)
        n8n_result = notify_n8n({**payload, "record_id": record_id})

        return jsonify({
            "ok": True,
            "record_id": record_id,
            **payload,
            "n8n": n8n_result
        })

    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500


@app.route("/analyze_voice", methods=["POST"])
def analyze_voice_api():
    try:
        customer_name = (request.form.get("customer_name") or "Unknown Customer").strip()
        audio_file = request.files.get("audio")

        payload = build_voice_payload(customer_name, audio_file)
        record_id = save_result(payload)
        n8n_result = notify_n8n({**payload, "record_id": record_id})

        return jsonify({
            "ok": True,
            "record_id": record_id,
            **payload,
            "n8n": n8n_result
        })

    except ValueError as e:
        return jsonify({"ok": False, "error": str(e)}), 400
    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500


In [7]:
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000).public_url
print("Public URL:", public_url)

def run_app():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

thread = Thread(target=run_app)
thread.daemon = True
thread.start()

print("Open this URL in your browser:")
print(public_url)


Public URL: https://mariyah-blearier-karen.ngrok-free.dev
Open this URL in your browser:
https://mariyah-blearier-karen.ngrok-free.dev


In [8]:
test_payload = {
    "customer_name": "Ahmed",
    "text": "الخدمة سيئة جدا وأنا غاضب جدا"
}

response = requests.post(f"{public_url}/analyze_text", json=test_payload)
response.json()


INFO:werkzeug:127.0.0.1 - - [06/May/2026 23:39:03] "POST /analyze_text HTTP/1.1" 200 -


{'confidence': 0.4762,
 'created_at': '2026-05-06T23:39:03.345732+00:00',
 'customer_name': 'Ahmed',
 'input_type': 'text',
 'n8n': {'message': 'N8N webhook not configured yet', 'sent': False},
 'ok': True,
 'original_text': 'الخدمة سيئة جدا وأنا غاضب جدا',
 'recommended_action': 'Escalate immediately to customer support',
 'record_id': 2,
 'sentiment': 'Angry',
 'sentiment_ar': 'غاضب',
 'transcribed_text': 'الخدمة سيئة جدا وأنا غاضب جدا'}

In [11]:
requests.get(f"{public_url}/history").json()


INFO:werkzeug:127.0.0.1 - - [06/May/2026 23:40:28] "GET /history HTTP/1.1" 200 -


[{'confidence': 0.4762,
  'created_at': '2026-05-06T23:39:03.345732+00:00',
  'customer_name': 'Ahmed',
  'id': 2,
  'input_type': 'text',
  'original_text': 'الخدمة سيئة جدا وأنا غاضب جدا',
  'recommended_action': 'Escalate immediately to customer support',
  'sentiment': 'Angry',
  'sentiment_ar': 'غاضب',
  'transcribed_text': 'الخدمة سيئة جدا وأنا غاضب جدا'},
 {'confidence': 0.4324,
  'created_at': '2026-05-06T23:38:28.573145+00:00',
  'customer_name': 'اسماء',
  'id': 1,
  'input_type': 'text',
  'original_text': 'هذا المنتج غير جيد',
  'recommended_action': 'Escalate immediately to customer support',
  'sentiment': 'Angry',
  'sentiment_ar': 'غاضب',
  'transcribed_text': 'هذا المنتج غير جيد'}]